In [1]:
import os
import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
from glob import glob
import numpy as np
from spectral.io import envi
from tqdm import tqdm

os.chdir('/store/carroll/sbgplants/')

In [21]:
# file paths
raw = 'data/raw'
rdn_fol = '/store/carroll/col/data/2018/raw/L1/'
out_folder = 'data/out_csv'

table = 'extracted_locations'

In [3]:
# load relevant output tables

pixel = pd.read_csv(os.path.join(out_folder, 'pixel.csv'))
fids = pixel.granule_id.unique()

In [16]:
# extract loc per px

fps = [x for x in glob(os.path.join(rdn_fol, '*/*rdn_ort_igm_ort.hdr')) if any(xx in x for xx in fids)]
id_cols = pixel.columns
val_cols = ['Lon', 'Lat', 'Elevation']

out = []

for fp in tqdm(fps):
    fid = fp.split('/')[-1].removesuffix('_rdn_ort_igm_ort.hdr')
    tmp = pixel[pixel['granule_id']==fid].copy()
    # extract loc
    loc = envi.open(fp).open_memmap()
    r = tmp['glt_row']; c=tmp['glt_column']
    vals = loc[r, c, :]
    # format df
    tmp = pd.concat([tmp, pd.DataFrame(vals, index=tmp.index, columns=val_cols)], axis=1)
    tmp = tmp.melt(id_vars=id_cols, value_vars=val_cols, var_name='loc_type', value_name='loc_value')
    out.append(tmp)

df = pd.concat(out)

100%|██████████████████████████████████████████████████████████████████████████████████████████| 61/61 [00:02<00:00, 21.46it/s]


In [17]:
df

,pixel_id,raster_plot_id,granule_id,glt_row,glt_column,loc_type,loc_value
0,1303,505,NIS01_20180612_175319,287,51,Lon,3.309864e+05
1,1304,505,NIS01_20180612_175319,287,52,Lon,3.309875e+05
2,1305,505,NIS01_20180612_175319,288,51,Lon,3.309860e+05
3,1306,505,NIS01_20180612_175319,288,52,Lon,3.309870e+05
4,1303,505,NIS01_20180612_175319,287,51,Lat,4.310048e+06
...,...,...,...,...,...,...,...
724,3799,360,NIS01_20180626_171026,879,5854,Elevation,2.828928e+03
725,3800,360,NIS01_20180626_171026,879,5855,Elevation,2.828673e+03
726,3801,360,NIS01_20180626_171026,880,5853,Elevation,2.830157e+03
727,3802,360,NIS01_20180626_171026,880,5854,Elevation,2.829907e+03


In [20]:
# prepare & populate out table
out_table = df.copy()

out_table['loc_id'] = range(len(out_table))
out_table = out_table[['loc_id','pixel_id','loc_type','loc_value']]

out_table

,loc_id,pixel_id,loc_type,loc_value
0,0,1303,Lon,3.309864e+05
1,1,1304,Lon,3.309875e+05
2,2,1305,Lon,3.309860e+05
3,3,1306,Lon,3.309870e+05
4,4,1303,Lat,4.310048e+06
...,...,...,...,...
724,45703,3799,Elevation,2.828928e+03
725,45704,3800,Elevation,2.828673e+03
726,45705,3801,Elevation,2.830157e+03
727,45706,3802,Elevation,2.829907e+03


In [22]:
# export table
fp_out = os.path.join(out_folder, f'{table}.csv')
out_table.to_csv(fp_out, index=False)